# Distance file PDF Pre-processing
This test file uses the Textract API to extract relevant details from receipts/invoices. The testing has been done on one file and pandas library has been used for faster delivery as it is a small file.


Importing all libraries and initialising the bucket and document to be processed


In [18]:
import json
import boto3
import io
import pandas as pd

bucket = 'cn01-project-input-205096516800-us-east-2-an'
document = 'TestFile2.pdf'

*analyze_document* textract library has been used to detect and extract the details needed for the use case. The output consists of the extracted data along with confidence scores. For the purpose of illustration, focus has been laid on printing the extracted data only.

In [19]:
s3_connection = boto3.resource('s3')
client = boto3.client('textract', region_name='us-east-2')

response = client.analyze_document(
        Document={
            'S3Object': {
                'Bucket': bucket,
                'Name': document
            }
        },
        FeatureTypes=['TABLES']
    )


In [12]:
blocks = response.get('Blocks', [])
block_map = {block['Id']: block for block in blocks}
# print(block_map)

tables = []

# Extract tables
for block in blocks:
    if block.get('BlockType') == 'TABLE':
        # print(block)
        
        cells = []
        for rel in block.get('Relationships', []):
            # print(rel)
            if rel.get('Type') == 'CHILD':
                # print(rel.get('Ids'))
                ids = rel.get('Ids', [])
                # print(ids)
                
                for cell_id in ids:
                    cell_block = block_map.get(cell_id)
                    # print(cell_block)
                    
                    if not cell_block or cell_block.get('BlockType') != 'CELL':
                        continue
                    
                    # Get cell text
                    cell_text = ""
                    for cell_rel in cell_block.get('Relationships', []):
                        if cell_rel.get('Type') == 'CHILD':
                            
                            word_ids = cell_rel.get('Ids', [])
                            # print(word_ids)
                            
                            for word_id in word_ids:
                                word = block_map.get(word_id)
                                if word and word.get('BlockType') == 'WORD':
                                    cell_text += word.get('Text', '') + ' '
                    
                    row = cell_block.get('RowIndex', 0)
                    col = cell_block.get('ColumnIndex', 0)
                    
                    cells.append({
                        'row': row,
                        'col': col,
                        'text': cell_text.strip(),
                    })
        
        if cells:
            tables.append(cells)
            print("Tables extracted")
                

  Tables extracted


In [13]:
table_array = [[''] * (col + 1) for _ in range(row + 1)]
        
# Fill array with cell text
for cell in cells:
    table_array[cell['row']][cell['col']] = cell['text']
        
    # Convert to DataFrame
df = pd.DataFrame(table_array)
# df.to_csv("s3://cn01-project-output-205096516800-us-east-2-an/processed_files/distancelog.csv")

In [17]:
pd.set_option('display.max_columns',None)
print(df.head(10))

  0        1         2                        3                 4       5   \
0                                                                            
1        DATE  START KM           STARTING POINT       DESTINATION  END KM   
2     02/5/22    771515           Abbotsford, BC     Valemount, BC  772250   
3     03/5/22    772250            Valemount, BC         Nisku, AB  772750   
4     04/5/22    772750  nisku to grande prairie         Nisku, AB  773800   
5     05/5/22    773800                Nisku, AB  Dawson Creek, BC  774430   
6     06/5/22    774430        Fort St. Jhon, BC         Nisku, AB  775250   
7     09/5/22    775250  nisku to grande prairie         Nisku, AB  776280   
8     10/5/22    776280  nisku to grande prairie         Nisku, AB  777300   
9     11/5/22    777300                Nisku, AB      Chetwynd, BC  778050   

         6       7       8       9       10      11      12      13       14  \
0                                                            